# 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import glob
import polars as pl
from pathlib import Path

DATA_PATH = Path("./data/UpdatedData/citibike_2023_combined.parquet/")

# 2. Data Load & Process

Processing script should be in `preprocessing.ipynb`

In [ ]:
df = (
    pl.read_parquet(f"{DATA_PATH}/*.parquet")
    # 1) Parse to datetime and rename in one go
    .with_columns(
        pl.col("started_at").str.to_datetime().alias("start_time"),
        pl.col("ended_at").str.to_datetime().alias("end_time"),
    )
    # 2) Compute ride_time_seconds = end_time - start_time (in seconds)
    .with_columns(
        (pl.col("end_time") - pl.col("start_time"))
        .dt.total_seconds()
        .alias("ride_time_seconds")
    )
    # 3) Order by start_time
    .sort("start_time")
    # 4) Select columns in the exact order you want
    .select(
        "ride_id",
        "rideable_type",
        "start_time",
        "end_time",
        "ride_time_seconds",   # ← right after end_time
        "start_station_name",
        "start_station_id",
        "end_station_name",
        "end_station_id",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng",
        "member_casual",
    )
)

# remove any start or end_station_name = null
df = df.filter(pl.col("end_station_name").is_not_null())
df= df.filter(pl.col("end_station_id").is_not_null())
df = df.filter(pl.col("start_station_name").is_not_null())
df= df.filter(pl.col("start_station_id").is_not_null())

# filter data out for ride_time < 2 hours or way too short ones like 1 minute
df = df.filter(pl.col("ride_time_seconds") < 2 * 3600)   
df = df.filter(pl.col("ride_time_seconds") >= 60)

df = df.filter(pl.col("start_time").dt.year() == 2023)
df = df.with_columns([
    pl.col("start_time").dt.month().alias("month"),
])

# filter data out for ride time that is negative or zero (not possible)
df = df.filter(pl.col("ride_time_seconds") > 0)

# Remove rides with identical start/end stations AND very short duration
df = df.filter(
    ~(
        (pl.col("start_station_id") == pl.col("end_station_id"))
        & (pl.col("ride_time_seconds") < 90)
    )
)

# 3. Visualization